<a href="https://colab.research.google.com/github/astrissha/Military-Engine-Failure-Prediction/blob/main/REC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

# Settings
num_records = 10000
num_engines = 10

# Start time
start_time = datetime.now()

data = []

for i in range(num_records):
    engine_id = random.randint(1, num_engines)

    # Generate a random timestamp
    timestamp = np.random.normal(150, 20)

    # Simulate sensor readings
    rpm = np.random.normal(1500, 200)
    oil_pressure = np.random.normal(60, 5)
    oil_temp = np.random.normal(90, 5)
    fuel_pressure = np.random.normal(45, 4)
    coolant_temp = np.random.normal(85, 3)
    vibration_level = np.random.normal(2, 0.5)
    exhaust_gas_temp = np.random.normal(450, 30)
    engine_load = np.random.normal(75, 10)
    ambient_temp = np.random.normal(25, 5)
    humidity = np.random.normal(50, 10)
    altitude = np.random.normal(300, 100)
    hours_since_maintenance = np.random.randint(0, 500)

    # Determine failure stage
    if (
        oil_pressure < 45 or oil_temp > 110 or vibration_level > 5 or
        exhaust_gas_temp > 550 or rpm > 1900
    ):
        failure_stage = "critical"
    elif (
        oil_pressure < 55 or oil_temp > 100 or vibration_level > 3.5 or
        exhaust_gas_temp > 500 or rpm > 1700
    ):
        failure_stage = "warning"
    else:
        failure_stage = "normal"

    data.append([
        timestamp, engine_id, rpm, oil_pressure, oil_temp, fuel_pressure,
        coolant_temp, vibration_level, exhaust_gas_temp, engine_load,
        ambient_temp, humidity, altitude, hours_since_maintenance, failure_stage
    ])

# Create DataFrame
columns = [
    "timestamp", "engine_id", "rpm", "oil_pressure", "oil_temp", "fuel_pressure",
    "coolant_temp", "vibration_level", "exhaust_gas_temp", "engine_load",
    "ambient_temp", "humidity", "altitude", "hours_since_maintenance", "failure_stage"
]

df = pd.DataFrame(data, columns=columns)

# Save to CSV
df.to_csv("synthetic_engine_failure_data.csv", index=False)
print("✅ Dataset generated with 'failure_stage' (normal, warning, critical)")


✅ Dataset generated with 'failure_stage' (normal, warning, critical)


In [ ]:
# Install necessary libraries
!pip install google-cloud-storage google-cloud-automl pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 5.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd

# Load your synthetic dataset (make sure it's uploaded to Colab or you can use Google Drive)
df = pd.read_csv('/content/synthetic_engine_failure_data.csv')
df.head()


,timestamp,engine_id,rpm,oil_pressure,oil_temp,fuel_pressure,coolant_temp,vibration_level,exhaust_gas_temp,engine_load,ambient_temp,humidity,altitude,hours_since_maintenance,failure_stage
0,150.713938,5,1682.888425,56.445567,82.934381,47.826220,90.538033,2.710221,435.302580,83.200111,27.845260,47.556063,118.068296,303,normal
1,169.738537,8,1697.202135,60.046868,101.611744,47.391819,81.229495,1.955170,457.620786,82.525241,26.418996,59.882147,174.696293,151,warning
2,162.319590,2,1023.229582,62.141080,92.212400,42.876544,86.902539,2.181814,430.650733,90.710436,23.671530,49.341831,217.795650,217,normal
3,124.054720,5,1352.619522,51.766529,95.048521,46.816319,76.587771,1.459005,424.676005,98.024244,20.741033,63.790486,284.094489,491,warning
4,142.562398,9,1279.343039,61.749759,92.533354,46.356031,86.108939,2.377118,405.814591,90.943560,22.051763,46.572023,176.625584,370,normal


In [ ]:
import pandas as pd


# Identify numerical and categorical columns
numerical_columns = df.select_dtypes(include=['number']).columns
categorical_columns = df.select_dtypes(include=['object']).columns

# Fill missing values for numerical columns with the mean
df[numerical_columns] = df[numerical_columns].fillna(df[numerical_columns].mean())

# Fill missing values for categorical columns with the mode (most frequent value)
for col in categorical_columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Now the dataset should be cleaned and ready for further processing
print(df.head())


    timestamp  engine_id          rpm  oil_pressure    oil_temp  \
0  150.713938          5  1682.888425     56.445567   82.934381   
1  169.738537          8  1697.202135     60.046868  101.611744   
2  162.319590          2  1023.229582     62.141080   92.212400   
3  124.054720          5  1352.619522     51.766529   95.048521   
4  142.562398          9  1279.343039     61.749759   92.533354   

   fuel_pressure  coolant_temp  vibration_level  exhaust_gas_temp  \
0      47.826220     90.538033         2.710221        435.302580   
1      47.391819     81.229495         1.955170        457.620786   
2      42.876544     86.902539         2.181814        430.650733   
3      46.816319     76.587771         1.459005        424.676005   
4      46.356031     86.108939         2.377118        405.814591   

   engine_load  ambient_temp   humidity    altitude  hours_since_maintenance  \
0    83.200111     27.845260  47.556063  118.068296                      303   
1    82.525241     26.

<ipython-input-4-f57be0a96af2>:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale numerical features
scaler = StandardScaler()
numerical_columns = ['timestamp','engine_id','oil_pressure', 'engine_load', 'vibration_level', 'humidity', 'coolant_temp','rpm','fuel_pressure','exhaust_gas_temp','ambient_temp','hours_since_maintenance','altitude']  # Example numerical features

df[numerical_columns] = scaler.fit_transform(df[numerical_columns])


In [ ]:
print(df.columns.tolist())



['timestamp', 'engine_id', 'rpm', 'oil_pressure', 'oil_temp', 'fuel_pressure', 'coolant_temp', 'vibration_level', 'exhaust_gas_temp', 'engine_load', 'ambient_temp', 'humidity', 'altitude', 'hours_since_maintenance', 'failure_stage']


In [ ]:
X = df.drop(columns=['failure_stage'])
y = df['failure_stage']
print(X)

      timestamp  engine_id       rpm  oil_pressure    oil_temp  fuel_pressure  \
0      0.041128  -0.194553  0.892556     -0.705184   82.934381       0.705324   
1      0.994436   0.845469  0.962775      0.018241  101.611744       0.597118   
2      0.622678  -1.234575 -2.343538      0.438924   92.212400      -0.527607   
3     -1.294745  -0.194553 -0.727647     -1.645105   95.048521       0.453765   
4     -0.367340   1.192143 -1.087120      0.360316   92.533354       0.339110   
...         ...        ...       ...           ...         ...            ...   
9995  -1.223288  -1.581249  0.955155     -0.673224   84.872335      -0.993652   
9996   1.429581   0.152121 -0.295345      0.332604   88.782475       0.874737   
9997   0.824810   0.845469  1.187325     -0.762710   90.195338       0.266879   
9998   1.425685  -0.887901 -0.696859     -0.297839   78.514598      -0.881588   
9999  -0.937944  -1.234575 -0.558991      0.064234   90.473368       0.179346   

      coolant_temp  vibrati

In [ ]:
!pip install flaml



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 6.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd

# Load your dataset (assuming it's in your Colab environment or Google Drive)
df = pd.read_csv('/content/synthetic_engine_failure_data.csv')  # use the new CSV name



# Fill missing numerical values with column mean
numerical_columns = df.select_dtypes(include=['number']).columns
df[numerical_columns] = df[numerical_columns].fillna(df[numerical_columns].mean())

# Encode categorical variables
categorical_columns = df.select_dtypes(include=['object']).columns

# Make sure not to encode the target yet
categorical_columns = [col for col in categorical_columns if col != 'failure_stage']

for col in categorical_columns:
    df[col] = df[col].astype('category').cat.codes  # Encode categorical columns

# Encode target variable (failure_stage) into numeric labels
df['failure_stage'] = df['failure_stage'].map({'normal': 0, 'warning': 1, 'critical': 2})





In [ ]:
# Uninstall numpy completely
!pip uninstall numpy -y

# Install compatible version of numpy
!pip install numpy==1.26.4 --quiet




Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
# Now that numpy is safe, install FLAML
!pip install flaml --upgrade


In [ ]:
from sklearn.model_selection import train_test_split
from flaml import AutoML
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Load dataset
df = pd.read_csv('/content/drive/MyDrive/synthetic_engine_failure_data.csv')

# Preprocessing
numerical_columns = df.select_dtypes(include=['number']).columns
df[numerical_columns] = df[numerical_columns].fillna(df[numerical_columns].mean())

categorical_columns = df.select_dtypes(include=['object']).columns
for col in categorical_columns:
    df[col] = df[col].astype('category').cat.codes

# Define X and y
X = df.drop('engine_health', axis=1)   # Replace 'engine_health' with your actual target column name
y = df['engine_health']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train AutoML
automl = AutoML()
automl.fit(X_train=X_train, y_train=y_train, task='classification', time_budget=60)

# Best model
best_model = automl.model

# Predictions
y_pred = best_model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.

In [ ]:
print("Best ML learner:", automl.best_estimator)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
le = LabelEncoder()

# Assuming `y_test` is in string format, we need to encode it
if y_test.dtype == 'object':
    y_test = le.fit_transform(y_test)  # Encode string labels to integers

# Predict the test set results
y_pred = best_model.predict(X_test)

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy of the model: {accuracy * 100:.2f}%')

# Decode the predictions (if necessary)
y_pred_labels = le.inverse_transform(y_pred)  # Decode predictions back to strings

# Print detailed classification report
print("Classification Report:")
print(classification_report(le.inverse_transform(y_test), y_pred_labels))  # Decode y_test if necessary



In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["normal", "warning", "critical"]))


In [ ]:
print(f"Predictions on test set: {y_pred}")
print(f"Predictions count by class: {pd.Series(y_pred).value_counts()}")


In [ ]:
from sklearn.metrics import accuracy_score


# Predict on training data
train_preds = best_model.predict(X_train)
train_acc = accuracy_score(y_train, train_preds)

# Predict on testing data
test_preds = best_model.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)

print(f"✅ Training Accuracy: {train_acc:.2f}")
print(f"✅ Testing Accuracy: {test_acc:.2f}")


In [ ]:
print(automl.best_config)   # Shows best hyperparameters
print(automl.best_estimator)  # Best model type (e.g., 'rf', 'xgboost')
print(automl.best_loss)     # Loss on validation set


In [ ]:
import joblib

joblib.dump(automl, 'smartguard_model.pkl')


In [ ]:
new_data = X_test.iloc[:5]  # Or any new synthetic/real sample
predictions = automl.predict(new_data)
print(predictions)


In [ ]:
probabilities = automl.predict_proba(new_data)
print(probabilities)


In [ ]:
def predict_failure(input_df):
    model = joblib.load("smartguard_model.pkl")
    return model.predict(input_df)


In [ ]:
pip install streamlit

In [ ]:
print(y_train.value_counts())  # or np.bincount(y_train) if it's a NumPy array


In [ ]:
%%writefile dig.py
import streamlit as st
import pandas as pd
import joblib
import plotly.graph_objects as go
import folium
import streamlit.components.v1 as components
from datetime import datetime

# Load trained model
automl = joblib.load("smartguard_model.pkl")

# Required feature columns
feature_columns = [
    'timestamp', 'engine_id', 'rpm', 'oil_pressure', 'oil_temp',
    'fuel_pressure', 'coolant_temp', 'vibration_level',
    'exhaust_gas_temp', 'engine_load', 'ambient_temp',
    'humidity', 'altitude', 'hours_since_maintenance',
    'latitude', 'longitude'
]

# Streamlit config and style
st.set_page_config(page_title="SMARTGUARD Digital Twin", layout="wide")

st.markdown("""
    <style>
    .stApp {
        background-color: #001f3f;
        color: white;
    }
    h1, h2, h3, label, .stMarkdown, .stNumberInput > label {
        color: white !important;
    }
    .stButton > button {
        background-color: #0074D9;
        color: white;
        border-radius: 5px;
        padding: 10px;
    }
    </style>
""", unsafe_allow_html=True)

# Logo
st.image("WhatsApp Image 2025-04-13 at 07.16.33.jpeg", width=120)

# Title and description
st.markdown("<h1 style='text-align: center;'>🚛 SMARTGUARD: Military Engine Digital Twin</h1>", unsafe_allow_html=True)
st.markdown("<p style='text-align: center; font-size:18px;'>Monitor, Predict & Visualize Military Engine Health Using AI</p>", unsafe_allow_html=True)

# Clock
current_time = datetime.now().strftime("%H:%M:%S")
st.markdown(f"<h4 style='text-align:right; color:gray;'>🕒 {current_time}</h4>", unsafe_allow_html=True)

# File uploader
st.subheader("📄 Upload Engine Sensor Data (Excel Format)")
uploaded_file = st.file_uploader("Upload .xlsx file with all required columns", type=["xlsx"])

# Prediction function
def digital_twin_prediction(df):
    df = df[feature_columns]
    prediction = automl.predict(df)
    return prediction

# Map function
def create_geolocation_map(lat, lon):
    m = folium.Map(location=[lat, lon], zoom_start=12)
    folium.Marker([lat, lon], popup=f"Latitude: {lat}, Longitude: {lon}").add_to(m)
    return m

if uploaded_file is not None:
    df_input = pd.read_excel(uploaded_file)

    if set(feature_columns).issubset(df_input.columns):
        preds = digital_twin_prediction(df_input)
        df_input['prediction'] = preds

        label_map = {0: "normal", 1: "warning", 2: "critical"}
        emoji_map = {"normal": "🟢 NORMAL", "warning": "🟡 WARNING", "critical": "🔴 CRITICAL"}

        for idx, row in df_input.iterrows():
            label = label_map.get(row['prediction'], "unknown")
            emoji = emoji_map.get(label, "❓ UNKNOWN")

            st.markdown(f"<h3>🛠️ Engine ID: {int(row['engine_id'])} - Health Status: {emoji}</h3>", unsafe_allow_html=True)

            # Show 3D Visualization
            fig = go.Figure()
            fig.add_trace(go.Scatter3d(
                x=[row['rpm']],
                y=[row['vibration_level']],
                z=[row['engine_load']],
                mode='markers',
                marker=dict(size=10, color='red'),
                name='Sensor'
            ))
            fig.update_layout(
                scene=dict(
                    xaxis_title='RPM',
                    yaxis_title='Vibration',
                    zaxis_title='Engine Load'
                ),
                margin=dict(l=0, r=0, b=0, t=0),
                height=400
            )
            st.plotly_chart(fig, use_container_width=True)

            # Show Location Map
            st.subheader("📍 Engine Location on Map")
            map_obj = create_geolocation_map(row['latitude'], row['longitude'])
            components.html(map_obj._repr_html_(), height=400)
            st.markdown("---")

    else:
        st.error("❌ Uploaded file must contain all required columns:\n\n" + ", ".join(feature_columns))
